In [32]:
# DIAGNOSTIC 1: Check if the 12 timestomped files have LogFile evidence

import pandas as pd

# Load raw LogFile
lf_raw = pd.read_csv('/Users/soni/Github/Digital-Detectives_Thesis/data/Lone Wolf/LW-LogFile.csv')

# The 12 known timestomped files
known_files = [
    'DeathToll.jpg', 'DemLogic.jpg', 'HoldMyTidePod.jpg', 
    'Planning.docx', 'Huckleberry.png', 'MyTiredHead.jpg',
    'Sheep.jpg', 'RedGuns.jpg', 'CubaDearmed.jpg',
    'AIRPORT INFORMATION.docx', 'DarkWolf.png', 'BladeofGrass.jpg'
]

print("="*80)
print("DIAGNOSTIC 1: SEARCHING RAW LOGFILE FOR THE 12 TIMESTOMPED FILES")
print("="*80)

total_with_logfile = 0
total_with_time_reversal = 0
total_with_zero_nano = 0

for filename in known_files:
    # Search for this file in LogFile (case-insensitive)
    if 'Filename' in lf_raw.columns:
        matches = lf_raw[lf_raw['Filename'].fillna('').str.contains(filename, case=False, na=False, regex=False)]
    else:
        # Try Full Path if Filename column doesn't exist
        matches = lf_raw[lf_raw['Full Path'].fillna('').str.contains(filename, case=False, na=False, regex=False)]
    
    print(f"\n{filename}:")
    print(f"  LogFile records found: {len(matches)}")
    
    if len(matches) > 0:
        total_with_logfile += 1
        
        # Check for Time Reversal or zero nanoseconds
        time_reversal = matches[matches['Event'].fillna('').str.contains('Time Reversal', case=False, na=False)]
        zero_nano = matches[matches['Detail'].fillna('').str.contains('Zero in 100-nanoseconds', case=False, na=False)]
        
        print(f"  Time Reversal events: {len(time_reversal)}")
        print(f"  Zero nanoseconds in Detail: {len(zero_nano)}")
        
        if len(time_reversal) > 0:
            total_with_time_reversal += 1
            print(f"  ✓✓✓ HAS TIME REVERSAL! ✓✓✓")
            
        if len(zero_nano) > 0:
            total_with_zero_nano += 1
            print(f"  ✓✓✓ HAS ZERO NANOSECONDS IN LOGFILE! ✓✓✓")
            
        if len(time_reversal) > 0 or len(zero_nano) > 0:
            print(f"  Events: {matches['Event'].unique()[:3]}")
            # Fixed: Check if Detail is not null before accessing
            detail_val = matches['Detail'].iloc[0]
            if pd.notna(detail_val) and len(str(detail_val)) > 0:
                print(f"  Sample Detail: {str(detail_val)[:150]}...")

print("\n" + "="*80)
print("SUMMARY")
print("="*80)
print(f"Files with ANY LogFile records: {total_with_logfile}/12")
print(f"Files with Time Reversal events: {total_with_time_reversal}/12")
print(f"Files with Zero nanoseconds in LogFile Detail: {total_with_zero_nano}/12")

if total_with_time_reversal > 0 or total_with_zero_nano > 0:
    print("\n✅ GOOD NEWS: LogFile evidence EXISTS for some files!")
    print("   → We can detect these files from raw CSV!")
else:
    print("\n❌ NO LogFile evidence found for any of the 12 files")
    print("   → These are UsnJrnl-only detections")


DIAGNOSTIC 1: SEARCHING RAW LOGFILE FOR THE 12 TIMESTOMPED FILES

DeathToll.jpg:
  LogFile records found: 4
  Time Reversal events: 1
  Zero nanoseconds in Detail: 1
  ✓✓✓ HAS TIME REVERSAL! ✓✓✓
  ✓✓✓ HAS ZERO NANOSECONDS IN LOGFILE! ✓✓✓
  Events: ['Move(After)' 'Updating MFTModified Time' 'Time Reversal Event']

DemLogic.jpg:
  LogFile records found: 4
  Time Reversal events: 1
  Zero nanoseconds in Detail: 1
  ✓✓✓ HAS TIME REVERSAL! ✓✓✓
  ✓✓✓ HAS ZERO NANOSECONDS IN LOGFILE! ✓✓✓
  Events: ['Move(After)' 'Updating MFTModified Time' 'Time Reversal Event']

HoldMyTidePod.jpg:
  LogFile records found: 4
  Time Reversal events: 1
  Zero nanoseconds in Detail: 1
  ✓✓✓ HAS TIME REVERSAL! ✓✓✓
  ✓✓✓ HAS ZERO NANOSECONDS IN LOGFILE! ✓✓✓
  Events: ['Move(After)' 'Updating MFTModified Time' 'Time Reversal Event']

Planning.docx:
  LogFile records found: 4
  Time Reversal events: 1
  Zero nanoseconds in Detail: 1
  ✓✓✓ HAS TIME REVERSAL! ✓✓✓
  ✓✓✓ HAS ZERO NANOSECONDS IN LOGFILE! ✓✓✓
  Events: ['

In [33]:
# DIAGNOSTIC 2: Check if training data has UsnJrnl-only timestomping examples

import pandas as pd
import os

training_data_path = '/Users/soni/Github/Digital-Detectives_Thesis/data/processed/Phase 2 - Features'

# Check if the combined training file exists
combined_file = f'{training_data_path}all_cases_combined.csv'

if os.path.exists(combined_file):
    print("="*80)
    print("DIAGNOSTIC 2: CHECKING TRAINING DATA FOR USNJRNL-ONLY EXAMPLES")
    print("="*80)
    
    # Load the combined training data
    train_df = pd.read_csv(combined_file, low_memory=False)
    
    print(f"Total training samples: {len(train_df):,}")
    
    # Check for ground truth labels
    if 'ground_truth_label' in train_df.columns:
        timestomped_count = (train_df['ground_truth_label'] == 1).sum()
        print(f"Timestomped files (ground_truth=1): {timestomped_count:,}")
        
        # Find UsnJrnl-only timestomped files
        if 'has_logfile_evidence' in train_df.columns and 'has_usnjrnl_evidence' in train_df.columns:
            usnjrnl_only_timestomped = train_df[
                (train_df['ground_truth_label'] == 1) & 
                (train_df['has_logfile_evidence'] == False) &
                (train_df['has_usnjrnl_evidence'] == True)
            ]
            
            print(f"UsnJrnl-only timestomped files: {len(usnjrnl_only_timestomped):,}")
            
            if len(usnjrnl_only_timestomped) > 0:
                print("\n✅ GOOD NEWS: Training data HAS UsnJrnl-only examples!")
                print(f"   → Model has seen {len(usnjrnl_only_timestomped)} examples of this pattern")
                
                # Show feature distribution
                print("\nFeature distribution in UsnJrnl-only timestomped files:")
                feature_cols = [
                    'basic_info_changed', 'path_depth', 'is_executable', 
                    'is_document', 'is_archive', 'is_image',
                    'in_temp_directory', 'in_system_directory'
                ]
                
                for col in feature_cols:
                    if col in usnjrnl_only_timestomped.columns:
                        if usnjrnl_only_timestomped[col].dtype == 'bool' or usnjrnl_only_timestomped[col].dtype == 'object':
                            true_count = (usnjrnl_only_timestomped[col] == True).sum()
                            true_pct = true_count / len(usnjrnl_only_timestomped) * 100
                            print(f"  {col}: {true_pct:.1f}% are True ({true_count}/{len(usnjrnl_only_timestomped)})")
                        else:
                            mean_val = usnjrnl_only_timestomped[col].mean()
                            print(f"  {col}: mean = {mean_val:.2f}")
                
                # Show some examples
                print("\nSample UsnJrnl-only timestomped files:")
                display_cols = ['filename', 'full_path', 'basic_info_changed']
                available_cols = [col for col in display_cols if col in usnjrnl_only_timestomped.columns]
                if available_cols:
                    print(usnjrnl_only_timestomped[available_cols].head(10).to_string(index=False))
                
            else:
                print("\n❌ BAD NEWS: NO UsnJrnl-only examples in training data!")
                print("   → Model never learned to detect UsnJrnl-only timestomping")
                print("   → This explains why it can't detect the 12 Lone Wolf files!")
        else:
            print("\n⚠️  WARNING: Training data missing has_logfile_evidence / has_usnjrnl_evidence columns")
            print("   → Cannot determine if UsnJrnl-only examples exist")
    else:
        print("\n⚠️  WARNING: Training data missing ground_truth_label column")
        print("   → Cannot analyze timestomped vs benign distribution")
        
else:
    print(f"❌ Training data file not found: {combined_file}")
    print("   → Cannot analyze training data composition")


❌ Training data file not found: /Users/soni/Github/Digital-Detectives_Thesis/data/processed/Phase 2 - Featuresall_cases_combined.csv
   → Cannot analyze training data composition


In [34]:
# DIAGNOSTIC 3: Identify which 3 of the 12 files were missed and WHY

import pandas as pd

# Load the datasets
suspicious = pd.read_csv('/Users/soni/Github/Digital-Detectives_Thesis/data/Lone Wolf/LW-Suspicious.csv')
data_features = pd.read_csv('/Users/soni/Github/Digital-Detectives_Thesis/data/processed/Prototype Tool Output/LW/data_features.csv')
flagged_files = pd.read_csv('/Users/soni/Github/Digital-Detectives_Thesis/data/processed/Prototype Tool Output/LW/flagged_files.csv')

print("="*80)
print("DIAGNOSTIC 3: ANALYZING THE 3 MISSED FILES")
print("="*80)

# Get the 12 known timestomped files from suspicious.csv
suspicious_timestomped = suspicious[suspicious['category'] == 'Timestamp Manipulation'].copy()

# Extract filenames from detail field
import re
suspicious_timestomped['filename'] = suspicious_timestomped['detail'].str.extract(r'timestamp of (.+?)"')[0]

known_files = suspicious_timestomped['filename'].tolist()
print(f"\nKnown timestomped files: {len(known_files)}")
for i, f in enumerate(known_files, 1):
    print(f"  {i}. {f}")

# Check which ones were detected
detected_files = flagged_files['filename'].tolist()
print(f"\nDetected files: {len(detected_files)}")

# Find the missed files
missed = [f for f in known_files if f not in detected_files]
detected = [f for f in known_files if f in detected_files]

print(f"\n✅ DETECTED ({len(detected)}):")
for f in detected:
    print(f"  ✓ {f}")

print(f"\n❌ MISSED ({len(missed)}):")
for f in missed:
    print(f"  ✗ {f}")

# Investigate each missed file
print("\n" + "="*80)
print("INVESTIGATING MISSED FILES")
print("="*80)

for filename in missed:
    print(f"\n{'='*60}")
    print(f"File: {filename}")
    print('='*60)
    
    # Find in data_features
    file_data = data_features[data_features['filename'] == filename]
    
    if len(file_data) == 0:
        print("  ❌ NOT FOUND in data_features.csv!")
        print("     → File was filtered out during merging/aggregation")
    else:
        row = file_data.iloc[0]
        print(f"  ✓ Found in data_features.csv")
        
        # Check key features
        key_features = [
            'zero_in_nanoseconds_combined',
            'zero_in_nanoseconds_suspicious', 
            'zero_in_nanoseconds_lf',
            'basic_info_changed',
            'has_usnjrnl_evidence',
            'has_logfile_evidence',
            'cross_artifact_validation_score',
            'time_reversal_event'
        ]
        
        print("\n  Features:")
        for feat in key_features:
            if feat in row.index:
                print(f"    {feat}: {row[feat]}")
        
        # Check if it has a prediction (might be below threshold)
        print(f"\n  USN: {row.get('usn_usn', 'N/A')}")
        print(f"  LSN: {row.get('lf_lsn', 'N/A')}")

# Also check predictions.csv to see if they got a confidence score below 70%
predictions_file = '/Users/soni/Github/Digital-Detectives_Thesis/data/processed/Prototype Tool Output/LW/predictions.csv'
if os.path.exists(predictions_file):
    print("\n" + "="*80)
    print("CHECKING PREDICTIONS FOR MISSED FILES")
    print("="*80)
    
    predictions = pd.read_csv(predictions_file)
    
    for filename in missed:
        pred = predictions[predictions['filename'] == filename]
        if len(pred) > 0:
            confidence = pred.iloc[0]['confidence_pct']
            print(f"\n{filename}:")
            print(f"  Confidence: {confidence:.2f}%")
            if confidence < 70:
                print(f"  ⚠️  BELOW THRESHOLD (70%)")
                print(f"  → Model detected it but confidence too low!")


DIAGNOSTIC 3: ANALYZING THE 3 MISSED FILES

Known timestomped files: 12
  1. DeathToll.jpg
  2. DemLogic.jpg
  3. HoldMyTidePod.jpg
  4. Planning.docx
  5. Huckleberry.png
  6. MyTiredHead.jpg
  7. Sheep.jpg
  8. RedGuns.jpg
  9. CubaDearmed.jpg
  10. AIRPORT INFORMATION.docx
  11. DarkWolf.png
  12. BladeofGrass.jpg

Detected files: 24

✅ DETECTED (12):
  ✓ DeathToll.jpg
  ✓ DemLogic.jpg
  ✓ HoldMyTidePod.jpg
  ✓ Planning.docx
  ✓ Huckleberry.png
  ✓ MyTiredHead.jpg
  ✓ Sheep.jpg
  ✓ RedGuns.jpg
  ✓ CubaDearmed.jpg
  ✓ AIRPORT INFORMATION.docx
  ✓ DarkWolf.png
  ✓ BladeofGrass.jpg

❌ MISSED (0):

INVESTIGATING MISSED FILES

CHECKING PREDICTIONS FOR MISSED FILES


In [35]:
# Add this to your diagnostic notebook

import pandas as pd

flagged = pd.read_csv('/Users/soni/Github/Digital-Detectives_Thesis/data/processed/Prototype Tool Output/LW/flagged_files.csv')
suspicious = pd.read_csv('/Users/soni/Github/Digital-Detectives_Thesis/data/Lone Wolf/LW-Suspicious.csv')

# Get known timestomped filenames
import re
suspicious_timestomped = suspicious[suspicious['category'] == 'Timestamp Manipulation'].copy()
suspicious_timestomped['filename'] = suspicious_timestomped['detail'].str.extract(r'timestamp of (.+?)"')[0]
known_files = suspicious_timestomped['filename'].tolist()

# Find additional flagged files (not in known list)
additional_flagged = flagged[~flagged['filename'].isin(known_files)]

print("="*80)
print(f"ADDITIONAL FLAGGED FILES: {len(additional_flagged)}")
print("="*80)

print("\nThese files were flagged by the model but NOT in suspicious.csv:")
for idx, row in additional_flagged.iterrows():
    print(f"\n{row['filename']}")
    print(f"  Confidence: {row['confidence_pct']:.2f}%")
    print(f"  Zero nanoseconds: {row.get('zero_in_nanoseconds', 'N/A')}")
    print(f"  Time Reversal: {row.get('time_reversal_event', 'N/A')}")
    print(f"  Cross-artifact score: {row.get('cross_artifact_validation_score', 'N/A')}")

# Check if they're in suspicious.csv under different categories
print("\n" + "="*80)
print("CHECKING IF ADDITIONAL FILES ARE IN SUSPICIOUS.CSV")
print("="*80)

for filename in additional_flagged['filename'].head(12):  # Check first 12
    matches = suspicious[suspicious['detail'].fillna('').str.contains(filename, case=False, na=False)]
    if len(matches) > 0:
        print(f"\n{filename}:")
        print(f"  Found in suspicious.csv!")
        print(f"  Category: {matches.iloc[0]['category']}")
        print(f"  Detail: {matches.iloc[0]['detail'][:100]}...")
    else:
        print(f"\n{filename}: NOT in suspicious.csv (potential false positive OR true positive missed by Oh et al.)")


ADDITIONAL FLAGGED FILES: 12

These files were flagged by the model but NOT in suspicious.csv:

appraiser.sdb
  Confidence: 99.84%
  Zero nanoseconds: True
  Time Reversal: True
  Cross-artifact score: 1.0

Set9A38.tmp
  Confidence: 99.83%
  Zero nanoseconds: False
  Time Reversal: True
  Cross-artifact score: 1.0

Set9A39.tmp
  Confidence: 99.83%
  Zero nanoseconds: False
  Time Reversal: True
  Cross-artifact score: 1.0

Set9A28.tmp
  Confidence: 99.83%
  Zero nanoseconds: False
  Time Reversal: True
  Cross-artifact score: 1.0

Appraiser_TelemetryRunList.xml
  Confidence: 99.82%
  Zero nanoseconds: True
  Time Reversal: True
  Cross-artifact score: 1.0

snapshot.etl
  Confidence: 99.80%
  Zero nanoseconds: False
  Time Reversal: True
  Cross-artifact score: 1.0

Appraiser_Data.ini
  Confidence: 99.72%
  Zero nanoseconds: True
  Time Reversal: True
  Cross-artifact score: 1.0

UDD9BA7.tmp
  Confidence: 99.64%
  Zero nanoseconds: False
  Time Reversal: True
  Cross-artifact score: 1.0

In [36]:
# Add to diagnostic notebook

import pandas as pd

# Load all flagged files with timestamps
flagged = pd.read_csv('/Users/soni/Github/Digital-Detectives_Thesis/data/processed/Prototype Tool Output/LW/flagged_files.csv')

# Get the 12 additional files
additional_files = [
    'appraiser.sdb', 'Set9A38.tmp', 'Set9A39.tmp', 'Set9A28.tmp',
    'Appraiser_TelemetryRunList.xml', 'snapshot.etl', 'Appraiser_Data.ini',
    'UDD9BA7.tmp', 'NVI2_29.DLL', 'Box Sync.lnk', 'Dropbox.lnk', 'Google Drive.lnk'
]

# Load data_features to get full details
data_features = pd.read_csv('/Users/soni/Github/Digital-Detectives_Thesis/data/processed/Prototype Tool Output/LW/data_features.csv')

print("="*80)
print("TIMELINE ANALYSIS OF ADDITIONAL FLAGGED FILES")
print("="*80)

for filename in additional_files:
    file_data = data_features[data_features['filename'] == filename]
    if len(file_data) > 0:
        row = file_data.iloc[0]
        print(f"\n{filename}:")
        print(f"  Full path: {row.get('full_path', 'N/A')}")
        print(f"  Event time: {row.get('lf_event_time', row.get('usn_event_time', 'N/A'))}")
        print(f"  LogFile Detail: {str(row.get('lf_detail', ''))[:100]}...")
        
# Compare to known timestomped files timeline
print("\n" + "="*80)
print("KNOWN TIMESTOMPED FILES TIMELINE (for comparison)")
print("="*80)

known_files = [
    'DeathToll.jpg', 'DemLogic.jpg', 'HoldMyTidePod.jpg', 'Planning.docx'
]

for filename in known_files[:4]:  # Show first 4 as examples
    file_data = data_features[data_features['filename'] == filename]
    if len(file_data) > 0:
        row = file_data.iloc[0]
        print(f"\n{filename}:")
        print(f"  Event time: {row.get('lf_event_time', row.get('usn_event_time', 'N/A'))}")
        print(f"  LogFile Detail: {str(row.get('lf_detail', ''))[:100]}...")


TIMELINE ANALYSIS OF ADDITIONAL FLAGGED FILES

appraiser.sdb:
  Full path: \Windows\appcompat\appraiser\AltData\appraiser.sdb
  Event time: 4/6/18 15:23
  LogFile Detail: ModifiedTime : 2018-04-06 15:23:03 -> 2018-03-23 04:22:26(Zero in 100-nanoseconds)...

Set9A38.tmp:
  Full path: \Users\jcloudy\AppData\Local\Temp\Set9A38.tmp
  Event time: 4/6/18 15:34
  LogFile Detail: ModifiedTime : 2018-04-06 15:34:31 -> 2018-04-06 15:24:33...

Set9A39.tmp:
  Full path: \Users\jcloudy\AppData\Local\Temp\Set9A39.tmp
  Event time: 4/6/18 15:34
  LogFile Detail: ModifiedTime : 2018-04-06 15:34:31 -> 2018-04-06 15:24:33...

Set9A28.tmp:
  Full path: \Users\jcloudy\AppData\Local\Temp\Set9A28.tmp
  Event time: 4/6/18 15:34
  LogFile Detail: ModifiedTime : 2018-04-06 15:34:31 -> 2018-04-06 15:24:33...

Appraiser_TelemetryRunList.xml:
  Full path: \Windows\appcompat\appraiser\AltData\Appraiser_TelemetryRunList.xml
  Event time: 4/6/18 15:23
  LogFile Detail: ModifiedTime : 2018-04-06 15:23:03 -> 2018-02-0

In [37]:
# Check production mode results

import pandas as pd

flagged = pd.read_csv('/Users/soni/Github/Digital-Detectives_Thesis/data/processed/Prototype Tool Output/LW/flagged_files.csv')

known_files = [
    'DeathToll.jpg', 'DemLogic.jpg', 'HoldMyTidePod.jpg', 
    'Planning.docx', 'Huckleberry.png', 'MyTiredHead.jpg',
    'Sheep.jpg', 'RedGuns.jpg', 'CubaDearmed.jpg',
    'AIRPORT INFORMATION.docx', 'DarkWolf.png', 'BladeofGrass.jpg'
]

detected = [f for f in known_files if f in flagged['filename'].values]

print("="*80)
print("PRODUCTION MODE RESULTS (NO suspicious.csv)")
print("="*80)
print(f"Known timestomped files detected: {len(detected)}/12")
print(f"Total files flagged: {len(flagged)}")

if len(detected) == 12:
    print("\n✅ SUCCESS! Model works in production mode!")
else:
    print(f"\n⚠️ Detected {len(detected)}/12 files")
    missed = [f for f in known_files if f not in flagged['filename'].values]
    print(f"Missed: {missed}")


PRODUCTION MODE RESULTS (NO suspicious.csv)
Known timestomped files detected: 12/12
Total files flagged: 24

✅ SUCCESS! Model works in production mode!


In [38]:
# Add this to diagnostic notebook

import pandas as pd

# Load suspicious.csv to get the EXACT LSN/USN numbers
suspicious = pd.read_csv('/Users/soni/Github/Digital-Detectives_Thesis/data/Lone Wolf/LW-Suspicious.csv')

# Get timestomped entries (UsnJrnl source)
suspicious_timestomped = suspicious[suspicious['category'] == 'Timestamp Manipulation'].copy()

print("="*80)
print("COMPARING BY LSN/USN (EXACT EVENT MATCH)")
print("="*80)

# Load flagged files
flagged = pd.read_csv('/Users/soni/Github/Digital-Detectives_Thesis/data/processed/Prototype Tool Output/LW/flagged_files.csv')

print("\nKnown timestomped USNs from suspicious.csv:")
known_usns = suspicious_timestomped['lsn/usn'].tolist()
for usn in known_usns:
    print(f"  USN: {usn}")

print(f"\nTotal known USNs: {len(known_usns)}")

# Check if these EXACT USNs are in flagged_files.csv
detected_by_usn = []
missed_by_usn = []

for usn in known_usns:
    if usn in flagged['usn_usn'].values:
        detected_by_usn.append(usn)
        # Get the filename for this USN
        filename = flagged[flagged['usn_usn'] == usn]['filename'].values[0]
        print(f"✓ USN {usn} detected ({filename})")
    else:
        missed_by_usn.append(usn)
        # Try to find filename from suspicious detail
        detail = suspicious_timestomped[suspicious_timestomped['lsn/usn'] == usn]['detail'].values[0]
        import re
        filename_match = re.search(r'timestamp of (.+?)"', detail)
        filename = filename_match.group(1) if filename_match else 'Unknown'
        print(f"✗ USN {usn} NOT detected ({filename})")
        
        # But check if the FILE was detected with a different USN
        if filename in flagged['filename'].values:
            actual_usn = flagged[flagged['filename'] == filename]['usn_usn'].values[0]
            print(f"  ⚠️  But file '{filename}' WAS detected with different USN: {actual_usn}")

print("\n" + "="*80)
print("RESULTS:")
print("="*80)
print(f"Exact USN matches: {len(detected_by_usn)}/{len(known_usns)}")
print(f"Missed USNs: {len(missed_by_usn)}/{len(known_usns)}")


COMPARING BY LSN/USN (EXACT EVENT MATCH)

Known timestomped USNs from suspicious.csv:
  USN: 239046272
  USN: 239049160
  USN: 239049856
  USN: 239050536
  USN: 239053344
  USN: 239054048
  USN: 239054704
  USN: 239055432
  USN: 239057384
  USN: 239057592
  USN: 239059120
  USN: 239059816

Total known USNs: 12
✗ USN 239046272 NOT detected (DeathToll.jpg)
  ⚠️  But file 'DeathToll.jpg' WAS detected with different USN: 249181568.0
✗ USN 239049160 NOT detected (DemLogic.jpg)
  ⚠️  But file 'DemLogic.jpg' WAS detected with different USN: 249180688.0
✗ USN 239049856 NOT detected (HoldMyTidePod.jpg)
  ⚠️  But file 'HoldMyTidePod.jpg' WAS detected with different USN: 249188064.0
✗ USN 239050536 NOT detected (Planning.docx)
  ⚠️  But file 'Planning.docx' WAS detected with different USN: 249191976.0
✗ USN 239053344 NOT detected (Huckleberry.png)
  ⚠️  But file 'Huckleberry.png' WAS detected with different USN: 249184256.0
✗ USN 239054048 NOT detected (MyTiredHead.jpg)
  ⚠️  But file 'MyTiredHea

In [39]:
# Add to diagnostic notebook - Check if the suspicious USNs even exist in our data

import pandas as pd

# Load raw UsnJrnl
usn_raw = pd.read_csv('/Users/soni/Github/Digital-Detectives_Thesis/data/Lone Wolf/LW-UsnJrnl.csv')

# Known USNs from suspicious.csv
known_usns = [239046272, 239049160, 239049856, 239050536, 239053344, 239054048, 
              239054704, 239055432, 239057384, 239057592, 239059120, 239059816]

print("="*80)
print("CHECKING IF SUSPICIOUS USNs EXIST IN RAW USNJRNL")
print("="*80)

for usn in known_usns:
    # Check if this USN exists in raw data
    match = usn_raw[usn_raw['USN'] == usn]
    
    if len(match) > 0:
        row = match.iloc[0]
        print(f"\n✓ USN {usn} EXISTS in raw UsnJrnl:")
        print(f"  Filename: {row.get('File/Directory Name', 'N/A')}")
        print(f"  EventInfo: {row.get('EventInfo', 'N/A')}")
        print(f"  Timestamp: {row.get('TimeStamp(UTC+8)', 'N/A')}")
        
        # Check if it has Basic_Info_Change
        if 'Basic_Info_Change' in str(row.get('EventInfo', '')):
            print(f"  ✓ Has Basic_Info_Change - SHOULD be filtered in")
        else:
            print(f"  ✗ NO Basic_Info_Change - WILL BE FILTERED OUT!")
    else:
        print(f"\n✗ USN {usn} NOT FOUND in raw UsnJrnl")

# Now check filtered data
print("\n" + "="*80)
print("CHECKING WHAT USNS WE ACTUALLY KEPT AFTER FILTERING")
print("="*80)

# Check data_merged.csv to see what USNs we have for these files
data_merged = pd.read_csv('/Users/soni/Github/Digital-Detectives_Thesis/data/processed/Prototype Tool Output/LW/data_merged.csv')

known_files = ['DeathToll.jpg', 'DemLogic.jpg', 'HoldMyTidePod.jpg', 'Planning.docx']

for filename in known_files:
    file_events = data_merged[data_merged['filename'] == filename]
    print(f"\n{filename}: {len(file_events)} events in data_merged.csv")
    if len(file_events) > 0:
        usn_values = file_events['usn_usn'].dropna().unique()
        print(f"  USN values: {sorted(usn_values)}")
        print(f"  USN range: {min(usn_values):.0f} to {max(usn_values):.0f}")


CHECKING IF SUSPICIOUS USNs EXIST IN RAW USNJRNL

✓ USN 239046272 EXISTS in raw UsnJrnl:
  Filename: DeathToll.jpg
  EventInfo: Basic_Info_Changed / File_Closed
  Timestamp: 4/5/18 10:21
  ✓ Has Basic_Info_Change - SHOULD be filtered in

✓ USN 239049160 EXISTS in raw UsnJrnl:
  Filename: DemLogic.jpg
  EventInfo: Basic_Info_Changed / File_Closed
  Timestamp: 4/5/18 10:21
  ✓ Has Basic_Info_Change - SHOULD be filtered in

✓ USN 239049856 EXISTS in raw UsnJrnl:
  Filename: HoldMyTidePod.jpg
  EventInfo: Basic_Info_Changed / File_Closed
  Timestamp: 4/5/18 10:21
  ✓ Has Basic_Info_Change - SHOULD be filtered in

✓ USN 239050536 EXISTS in raw UsnJrnl:
  Filename: Planning.docx
  EventInfo: Basic_Info_Changed / File_Closed
  Timestamp: 4/5/18 10:21
  ✓ Has Basic_Info_Change - SHOULD be filtered in

✓ USN 239053344 EXISTS in raw UsnJrnl:
  Filename: Huckleberry.png
  EventInfo: Basic_Info_Changed / File_Closed
  Timestamp: 4/5/18 10:21
  ✓ Has Basic_Info_Change - SHOULD be filtered in

✓ USN

In [40]:
import pandas as pd

# Check what columns we have in data_features.csv
data_features = pd.read_csv('/Users/soni/Github/Digital-Detectives_Thesis/data/processed/Prototype Tool Output/LW/data_features.csv')

print("="*80)
print("AVAILABLE COLUMNS IN data_features.csv")
print("="*80)
print(f"Total columns: {len(data_features.columns)}\n")

# Group columns by category
timestamp_cols = [col for col in data_features.columns if 'time' in col.lower() or 'event' in col.lower()]
detail_cols = [col for col in data_features.columns if 'detail' in col.lower()]
path_cols = [col for col in data_features.columns if 'path' in col.lower() or 'filename' in col.lower()]

print("Timestamp/Event columns:")
for col in timestamp_cols:
    print(f"  - {col}")

print("\nDetail/Description columns:")
for col in detail_cols:
    print(f"  - {col}")

print("\nFile path columns:")
for col in path_cols:
    print(f"  - {col}")

# Check a sample row for DeathToll.jpg
print("\n" + "="*80)
print("SAMPLE FORENSIC DETAILS FOR DeathToll.jpg")
print("="*80)

sample = data_features[data_features['filename'] == 'DeathToll.jpg'].iloc[0]

important_cols = [
    'filename', 'full_path', 
    'lf_event_time', 'usn_event_time',
    'lf_event', 'lf_detail',
    'usn_event_info',
    'lf_creation_time', 'lf_modified_time', 'lf_accessed_time',
    'confidence_pct'
]

available_cols = [col for col in important_cols if col in sample.index]

for col in available_cols:
    value = sample[col]
    if pd.notna(value):
        if isinstance(value, str) and len(str(value)) > 100:
            print(f"\n{col}:")
            print(f"  {str(value)[:200]}...")
        else:
            print(f"{col}: {value}")


AVAILABLE COLUMNS IN data_features.csv
Total columns: 64

Timestamp/Event columns:
  - EventTime(UTC+8)
  - lf_event
  - lf_creation_time
  - lf_modified_time
  - lf_mft_modified_time
  - lf_accessed_time
  - usn_event_time
  - usn_event_info
  - lf_event_count
  - usn_event_count
  - time_reversal_event
  - timestamp_changed_to_past
  - modified_creationtime
  - modified_modifiedtime
  - modified_mftmodifiedtime
  - modified_accessedtime
  - using_another_timestamp
  - si_timestamp_changed
  - fn_timestamp_changed
  - zero_nano_time_reversal
  - timestamp_equality

Detail/Description columns:
  - lf_detail
  - suspicious_detail

File path columns:
  - lf_full_path
  - usn_filename
  - usn_full_path
  - filename
  - full_path
  - path_depth
  - filename_length

SAMPLE FORENSIC DETAILS FOR DeathToll.jpg
filename: DeathToll.jpg
full_path: \Users\jcloudy\Dropbox\DeathToll.jpg
usn_event_time: 4/6/18 20:35
lf_event: Time Reversal Event
lf_detail: ModifiedTime : 2018-04-06 20:35:25 -> 2018-0